# Phase 1: Data Infrastructure & Empirical Diagnostics  
## Notebook 01_03 — Continuous Futures Contract Rolling

### Research question

How can we construct a continuous futures price series from individual expiring contracts without introducing artificial jumps, look-ahead bias, or misleading strategy returns?

This notebook builds a reproducible futures-rolling pipeline using synthetic contract data.

It covers:

1. why futures contracts cannot simply be concatenated;
2. how to generate individual expiring futures contracts;
3. calendar-based roll rules;
4. volume/liquidity-aware roll rules;
5. raw spliced continuous contracts;
6. backward additive adjustment;
7. backward ratio adjustment;
8. perpetual / weighted roll construction;
9. roll gap and roll yield diagnostics;
10. validation checks for continuity and point-in-time safety;
11. limitations and current research/industry directions.

The goal is not to claim one universal best continuous contract. The goal is to make the construction assumptions explicit, testable, and visible.

## 1. Why continuous futures are difficult

A futures contract has an expiry date. A crude oil, copper, soybean meal, equity index, or government bond futures strategy cannot trade the same contract forever.

A backtest therefore needs to decide:

1. **which contract to hold at each date**;
2. **when to roll from the expiring contract to the next contract**;
3. **how to represent historical prices after rolling**.

A naïve approach is to splice contracts together:

$$
P_t^{\text{raw}} =
\begin{cases}
P_t^{(1)}, & t < \tau_1 \\
P_t^{(2)}, & \tau_1 \leq t < \tau_2 \\
P_t^{(3)}, & \tau_2 \leq t < \tau_3
\end{cases}
$$
where $\tau_i$ are roll dates.

The problem is that two contracts with different expiries usually trade at different prices on the same date.

So raw splicing can create artificial jumps that were not actual market moves.

## 2. The futures curve and roll gap

Let:

- $F_t^{\text{old}}$ be the price of the expiring contract on the roll date;
- $F_t^{\text{new}}$ be the price of the next contract on the same roll date.

The roll gap is:

$$
g_t = F_t^{\text{new}} - F_t^{\text{old}}
$$
If:

$$
F_t^{\text{new}} > F_t^{\text{old}}
$$
the market is locally in **contango** around that roll.

If:

$$
F_t^{\text{new}} < F_t^{\text{old}}
$$
the market is locally in **backwardation** around that roll.

For a long futures position, the approximate roll return from closing the old contract and entering the new contract is:

$$
\begin{aligned}
\text{roll return}_t &\approx \frac{F_t^{\text{old}} - F_t^{\text{new}}}{F_t^{\text{old}}}
\end{aligned}
$$
This is not a price change in the underlying commodity. It is a return component caused by moving along the futures curve.

## 3. Imports and configuration

The notebook is fully reproducible and uses synthetic futures contracts.

No external market data is required.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)

## 4. Futures month codes

Many futures markets identify contracts using month codes.

This notebook uses standard month codes:

| Month | Code |
|---:|:---:|
| January | F |
| February | G |
| March | H |
| April | J |
| May | K |
| June | M |
| July | N |
| August | Q |
| September | U |
| October | V |
| November | X |
| December | Z |

For example, a synthetic March 2024 contract can be labelled:

```text
SYNH24
```

In [ ]:
MONTH_CODES = {
    1: "F",
    2: "G",
    3: "H",
    4: "J",
    5: "K",
    6: "M",
    7: "N",
    8: "Q",
    9: "U",
    10: "V",
    11: "X",
    12: "Z"
}


def contract_code(root: str, expiry: pd.Timestamp) -> str:
    """
    Create a futures contract code such as SYNH24.
    """
    month_code = MONTH_CODES[int(expiry.month)]
    year_code = str(expiry.year)[-2:]
    return f"{root}{month_code}{year_code}"

## 5. Synthetic futures contract generation

We generate:

1. a latent spot-like index;
2. a time-varying carry rate;
3. monthly futures contracts with different expiries;
4. contract-specific prices using an approximate cost-of-carry relationship:

$$
F(t,T) \approx S_t \exp(c_t (T-t))
$$
where:

- $S_t$ is the latent spot-like index;
- $c_t$ is the carry rate;
- $T-t$ is time to expiry in years.

This is simplified, but it creates realistic differences between contracts at the same date.

In [ ]:
def third_friday(year: int, month: int) -> pd.Timestamp:
    """
    Return the third Friday of a given month, normalised to UTC.
    """
    dates = pd.date_range(
        start=f"{year}-{month:02d}-01",
        end=f"{year}-{month:02d}-28",
        freq="D",
        tz="UTC"
    )
    fridays = dates[dates.weekday == 4]
    return fridays[2]


def make_monthly_expiries(start_year: int, end_year: int) -> list[pd.Timestamp]:
    """
    Generate monthly third-Friday expiries.
    """
    expiries = []

    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            expiries.append(third_friday(year, month))

    return expiries

In [ ]:
def simulate_latent_spot_and_carry(
    dates: pd.DatetimeIndex,
    s0: float = 100.0,
    annual_vol: float = 0.25
) -> pd.DataFrame:
    """
    Simulate a latent spot-like index and a time-varying carry rate.
    """
    n = len(dates)
    dt = 1 / 252

    shocks = annual_vol * np.sqrt(dt) * rng.standard_normal(n)
    drift = 0.03 * dt

    log_spot = np.log(s0) + np.cumsum(drift + shocks)
    spot = np.exp(log_spot)

    # Carry alternates between contango and backwardation regimes.
    time_index = np.arange(n)
    carry = (
        0.05 * np.sin(2 * np.pi * time_index / 260)
        + 0.02 * np.sin(2 * np.pi * time_index / 90)
        + 0.01 * rng.standard_normal(n)
    )

    return pd.DataFrame({
        "timestamp": dates,
        "spot_index": spot,
        "carry_rate": carry
    })

In [ ]:
@dataclass(frozen=True)
class FuturesUniverseConfig:
    """
    Synthetic futures universe configuration.
    """
    root: str = "SYN"
    start: str = "2022-01-01"
    end: str = "2024-12-31"
    min_days_to_expiry: int = 1
    max_days_to_expiry: int = 180
    currency: str = "USD"
    contract_multiplier: float = 100.0


def simulate_futures_contracts(config: FuturesUniverseConfig) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Simulate individual futures contracts with monthly expiries.

    Returns
    -------
    contracts:
        Long-format futures contract OHLCV data.

    latent:
        Latent spot and carry data.
    """
    dates = pd.date_range(config.start, config.end, freq="B", tz="UTC")
    latent = simulate_latent_spot_and_carry(dates)

    expiries = make_monthly_expiries(
        start_year=pd.Timestamp(config.start).year,
        end_year=pd.Timestamp(config.end).year + 1
    )

    frames = []

    for expiry in expiries:
        contract = contract_code(config.root, expiry)

        active = latent.copy()
        active["expiry"] = expiry
        active["days_to_expiry"] = (expiry - active["timestamp"]).dt.days

        active = active[
            (active["days_to_expiry"] >= config.min_days_to_expiry)
            & (active["days_to_expiry"] <= config.max_days_to_expiry)
        ].copy()

        if active.empty:
            continue

        tau = active["days_to_expiry"].to_numpy() / 365.0
        spot = active["spot_index"].to_numpy()
        carry = active["carry_rate"].to_numpy()

        micro_noise = rng.normal(loc=0.0, scale=0.0015, size=len(active))
        close = spot * np.exp(carry * tau + micro_noise)

        open_noise = rng.normal(loc=0.0, scale=0.002, size=len(active))
        open_price = close * np.exp(open_noise)

        high_spread = np.abs(rng.normal(loc=0.003, scale=0.002, size=len(active)))
        low_spread = np.abs(rng.normal(loc=0.003, scale=0.002, size=len(active)))

        high = np.maximum(open_price, close) * (1 + high_spread)
        low = np.minimum(open_price, close) * (1 - low_spread)

        # Volume rises after listing, peaks before expiry, then drops into expiry.
        dte = active["days_to_expiry"].to_numpy()
        volume_shape = np.exp(-((dte - 35) ** 2) / (2 * 35 ** 2))
        volume_noise = rng.lognormal(mean=0.0, sigma=0.25, size=len(active))
        volume = 50_000 * volume_shape * volume_noise + 1_000

        active["root"] = config.root
        active["contract"] = contract
        active["open"] = open_price
        active["high"] = high
        active["low"] = low
        active["close"] = close
        active["volume"] = volume
        active["currency"] = config.currency
        active["contract_multiplier"] = config.contract_multiplier

        frames.append(active[[
            "root",
            "contract",
            "timestamp",
            "expiry",
            "days_to_expiry",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "currency",
            "contract_multiplier",
            "spot_index",
            "carry_rate"
        ]])

    contracts = pd.concat(frames, ignore_index=True)
    contracts = contracts.sort_values(["contract", "timestamp"]).reset_index(drop=True)

    return contracts, latent

In [ ]:
config = FuturesUniverseConfig()
contracts, latent = simulate_futures_contracts(config)

contracts.head()

In [ ]:
contracts[["contract", "timestamp", "expiry", "days_to_expiry", "close", "volume"]].head()

## 6. Visualising individual contract prices

Each futures contract has its own price path.

A continuous contract is not a directly traded instrument. It is a research construction built from these individual contracts.

In [ ]:
sample_contracts = contracts["contract"].drop_duplicates().iloc[3:15].tolist()

plt.figure(figsize=(12, 6))

for contract in sample_contracts:
    part = contracts[contracts["contract"] == contract]
    plt.plot(part["timestamp"], part["close"], alpha=0.75, label=contract)

plt.title("Synthetic Individual Futures Contract Prices")
plt.xlabel("Date")
plt.ylabel("Futures price")
plt.legend(ncol=3, fontsize=8)
plt.show()

## 7. Roll schedule: calendar rule

A simple roll rule is:

> Hold the nearest contract whose expiry is at least $k$ business days away.

This avoids trading too close to expiry.

For example, with $k=5$, the system rolls out of the front contract five business days before expiry.

This rule is deterministic and easy to reproduce, but it may not match actual liquidity migration.

In [ ]:
def build_calendar_roll_schedule(
    contracts: pd.DataFrame,
    roll_days_before_expiry: int = 5
) -> pd.DataFrame:
    """
    Build a daily roll schedule using a calendar roll rule.

    Select the nearest contract whose expiry is at least roll_days_before_expiry
    calendar days away.
    """
    all_dates = pd.DatetimeIndex(sorted(contracts["timestamp"].unique()))
    rows = []

    for timestamp in all_dates:
        live = contracts[
            (contracts["timestamp"] == timestamp)
            & (contracts["days_to_expiry"] > roll_days_before_expiry)
        ]

        if live.empty:
            continue

        selected = live.sort_values("expiry").iloc[0]

        rows.append({
            "timestamp": timestamp,
            "selected_contract": selected["contract"],
            "selected_expiry": selected["expiry"],
            "days_to_expiry": selected["days_to_expiry"],
            "roll_rule": f"calendar_{roll_days_before_expiry}d"
        })

    schedule = pd.DataFrame(rows)
    schedule = schedule.sort_values("timestamp").reset_index(drop=True)

    return schedule


calendar_schedule = build_calendar_roll_schedule(
    contracts,
    roll_days_before_expiry=5
)

calendar_schedule.head()

In [ ]:
def extract_roll_events(schedule: pd.DataFrame) -> pd.DataFrame:
    """
    Extract roll events from a selected-contract schedule.
    """
    out = schedule.copy()
    out["previous_contract"] = out["selected_contract"].shift(1)
    out["is_roll"] = out["selected_contract"] != out["previous_contract"]

    events = out[out["is_roll"] & out["previous_contract"].notna()].copy()
    events = events.rename(columns={"selected_contract": "new_contract"})
    events = events[[
        "timestamp",
        "previous_contract",
        "new_contract",
        "selected_expiry",
        "days_to_expiry",
        "roll_rule"
    ]]

    return events.reset_index(drop=True)


calendar_roll_events = extract_roll_events(calendar_schedule)
calendar_roll_events.head()

## 8. Roll schedule: volume rule

A liquidity-aware rule is:

> Hold the live contract with the highest volume.

This better resembles how real traders often care about liquidity, but it can create instability if volume briefly flips between contracts.

In production, volume-based rolling usually needs smoothing and anti-flicker rules.

In [ ]:
def build_volume_roll_schedule(
    contracts: pd.DataFrame,
    min_days_to_expiry: int = 3
) -> pd.DataFrame:
    """
    Build a daily roll schedule by selecting the live contract with highest volume.

    To avoid holding contracts too close to expiry, contracts with days_to_expiry
    <= min_days_to_expiry are excluded.
    """
    all_dates = pd.DatetimeIndex(sorted(contracts["timestamp"].unique()))
    rows = []

    current_contract = None

    for timestamp in all_dates:
        live = contracts[
            (contracts["timestamp"] == timestamp)
            & (contracts["days_to_expiry"] > min_days_to_expiry)
        ].copy()

        if live.empty:
            continue

        selected = live.sort_values(["volume", "expiry"], ascending=[False, True]).iloc[0]

        # Anti-flicker rule:
        # once the selected contract has moved to a later expiry, do not move backwards.
        if current_contract is not None:
            current_expiry = contracts.loc[
                contracts["contract"] == current_contract,
                "expiry"
            ].iloc[0]

            candidate_expiry = selected["expiry"]

            if candidate_expiry < current_expiry:
                still_live_current = live[live["contract"] == current_contract]
                if not still_live_current.empty:
                    selected = still_live_current.iloc[0]

        current_contract = selected["contract"]

        rows.append({
            "timestamp": timestamp,
            "selected_contract": selected["contract"],
            "selected_expiry": selected["expiry"],
            "days_to_expiry": selected["days_to_expiry"],
            "roll_rule": "highest_volume"
        })

    schedule = pd.DataFrame(rows)
    schedule = schedule.sort_values("timestamp").reset_index(drop=True)

    return schedule


volume_schedule = build_volume_roll_schedule(contracts, min_days_to_expiry=3)
volume_roll_events = extract_roll_events(volume_schedule)

volume_roll_events.head()

## 9. Constructing the raw spliced continuous series

The raw spliced series simply selects the price of the scheduled contract on each date.

It is easy to construct, but it can contain artificial jumps at roll dates.

In [ ]:
def build_raw_continuous_series(
    contracts: pd.DataFrame,
    schedule: pd.DataFrame,
    price_column: str = "close"
) -> pd.DataFrame:
    """
    Build a raw spliced continuous futures series from a roll schedule.
    """
    selected = schedule.merge(
        contracts[[
            "timestamp",
            "contract",
            "expiry",
            "days_to_expiry",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "spot_index",
            "carry_rate"
        ]],
        left_on=["timestamp", "selected_contract"],
        right_on=["timestamp", "contract"],
        how="left"
    )

    selected = selected.rename(columns={
        price_column: "raw_price",
        "volume": "selected_volume"
    })

    selected = selected.sort_values("timestamp").reset_index(drop=True)

    selected["raw_return"] = selected["raw_price"].pct_change()
    selected["selected_contract_prev"] = selected["selected_contract"].shift(1)
    selected["is_roll"] = selected["selected_contract"] != selected["selected_contract_prev"]

    return selected


raw_calendar = build_raw_continuous_series(contracts, calendar_schedule)
raw_volume = build_raw_continuous_series(contracts, volume_schedule)

raw_calendar.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(raw_calendar["timestamp"], raw_calendar["raw_price"], label="Raw calendar-spliced series")
plt.scatter(
    raw_calendar.loc[raw_calendar["is_roll"], "timestamp"],
    raw_calendar.loc[raw_calendar["is_roll"], "raw_price"],
    s=15,
    label="Roll dates"
)
plt.title("Raw Spliced Continuous Futures Series")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.show()

## 10. Measuring roll gaps

At a roll date, both the old and new contracts usually exist.

So we can compare:

$$
F_t^{\text{old}}
\quad \text{and} \quad
F_t^{\text{new}}
$$
on the same timestamp.

This lets us measure the artificial jump introduced by raw splicing.

In [ ]:
def compute_roll_gap_table(
    contracts: pd.DataFrame,
    roll_events: pd.DataFrame
) -> pd.DataFrame:
    """
    Compute old/new prices and roll gaps on roll dates.
    """
    rows = []

    for _, event in roll_events.iterrows():
        timestamp = event["timestamp"]
        old_contract = event["previous_contract"]
        new_contract = event["new_contract"]

        old_row = contracts[
            (contracts["timestamp"] == timestamp)
            & (contracts["contract"] == old_contract)
        ]

        new_row = contracts[
            (contracts["timestamp"] == timestamp)
            & (contracts["contract"] == new_contract)
        ]

        if old_row.empty or new_row.empty:
            continue

        old_price = float(old_row["close"].iloc[0])
        new_price = float(new_row["close"].iloc[0])
        gap = new_price - old_price

        rows.append({
            "timestamp": timestamp,
            "old_contract": old_contract,
            "new_contract": new_contract,
            "old_price": old_price,
            "new_price": new_price,
            "roll_gap": gap,
            "roll_gap_pct_old": gap / old_price,
            "long_roll_return_approx": (old_price - new_price) / old_price,
            "term_structure_state": "contango" if gap > 0 else "backwardation"
        })

    return pd.DataFrame(rows)


calendar_gap_table = compute_roll_gap_table(contracts, calendar_roll_events)
calendar_gap_table.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(calendar_gap_table["timestamp"], calendar_gap_table["roll_gap_pct_old"])
plt.axhline(0, linewidth=1)
plt.title("Roll Gaps as Percentage of Old Contract Price")
plt.xlabel("Roll date")
plt.ylabel("Roll gap / old price")
plt.show()

## 11. Backward additive adjustment

The additive, or Panama-style, backward adjustment shifts historical prices by the roll gap.

At a roll date:

$$
g_t = F_t^{\text{new}} - F_t^{\text{old}}
$$
All prices before the roll date are shifted by $g_t$:

$$
P_s^{\text{adjusted}} = P_s^{\text{raw}} + g_t
\quad \text{for } s < t
$$
When multiple rolls occur, the historical adjustments accumulate.

This removes roll jumps from the continuous series, but it can distort the absolute historical price level and can sometimes create negative historical prices.

In [ ]:
def backward_additive_adjustment(
    raw_series: pd.DataFrame,
    gap_table: pd.DataFrame
) -> pd.DataFrame:
    """
    Apply backward additive adjustment to a raw continuous futures series.
    """
    adjusted = raw_series.copy()
    adjusted["back_adjusted_additive"] = adjusted["raw_price"].astype(float)

    for _, roll in gap_table.sort_values("timestamp").iterrows():
        roll_date = roll["timestamp"]
        gap = roll["roll_gap"]

        mask = adjusted["timestamp"] < roll_date
        adjusted.loc[mask, "back_adjusted_additive"] += gap

    adjusted["additive_return"] = adjusted["back_adjusted_additive"].pct_change()

    return adjusted


additive_calendar = backward_additive_adjustment(raw_calendar, calendar_gap_table)

additive_calendar[["timestamp", "raw_price", "back_adjusted_additive"]].head()

## 12. Backward ratio adjustment

The ratio adjustment multiplies historical prices by the ratio between new and old contract prices at the roll date:

$$
q_t = \frac{F_t^{\text{new}}}{F_t^{\text{old}}}
$$
For all $s < t$:

$$
P_s^{\text{adjusted}} = q_t P_s^{\text{raw}}
$$
This preserves proportional returns better than additive adjustment and avoids negative adjusted prices, but it still creates a synthetic price history rather than an actually tradable contract.

In [ ]:
def backward_ratio_adjustment(
    raw_series: pd.DataFrame,
    gap_table: pd.DataFrame
) -> pd.DataFrame:
    """
    Apply backward ratio adjustment to a raw continuous futures series.
    """
    adjusted = raw_series.copy()
    adjusted["back_adjusted_ratio"] = adjusted["raw_price"].astype(float)

    for _, roll in gap_table.sort_values("timestamp").iterrows():
        roll_date = roll["timestamp"]
        ratio = roll["new_price"] / roll["old_price"]

        mask = adjusted["timestamp"] < roll_date
        adjusted.loc[mask, "back_adjusted_ratio"] *= ratio

    adjusted["ratio_return"] = adjusted["back_adjusted_ratio"].pct_change()

    return adjusted


ratio_calendar = backward_ratio_adjustment(raw_calendar, calendar_gap_table)

ratio_calendar[["timestamp", "raw_price", "back_adjusted_ratio"]].head()

## 13. Perpetual / weighted roll construction

A perpetual series blends old and new contracts over a roll window.

For a roll window of $K$ days, the continuous price can be:

$$
\begin{aligned}
P_t^{\text{perpetual}} &= (1-\alpha_t)F_t^{\text{old}} \\
&\quad + \alpha_t F_t^{\text{new}}
\end{aligned}
$$
where $\alpha_t$ increases from 0 to 1 over the roll window.

This method is useful because it resembles gradually rolling a position.

However, it embeds a trading assumption: the backtest is no longer holding one contract at a time during the roll window.

In [ ]:
def build_perpetual_weighted_series(
    contracts: pd.DataFrame,
    raw_series: pd.DataFrame,
    gap_table: pd.DataFrame,
    roll_window: int = 5
) -> pd.DataFrame:
    """
    Build a linear weighted perpetual futures series around each roll date.
    """
    out = raw_series.copy()
    out["perpetual_price"] = out["raw_price"].astype(float)
    out["perpetual_old_weight"] = 1.0
    out["perpetual_new_weight"] = 0.0

    timestamps = pd.DatetimeIndex(out["timestamp"])

    for _, roll in gap_table.iterrows():
        roll_date = roll["timestamp"]
        old_contract = roll["old_contract"]
        new_contract = roll["new_contract"]

        roll_position = np.where(timestamps == roll_date)[0]

        if len(roll_position) == 0:
            continue

        roll_idx = int(roll_position[0])
        start_idx = max(0, roll_idx - roll_window + 1)
        window_indices = list(range(start_idx, roll_idx + 1))

        for local_k, idx in enumerate(window_indices):
            alpha = (local_k + 1) / len(window_indices)
            timestamp = out.loc[idx, "timestamp"]

            old_row = contracts[
                (contracts["timestamp"] == timestamp)
                & (contracts["contract"] == old_contract)
            ]

            new_row = contracts[
                (contracts["timestamp"] == timestamp)
                & (contracts["contract"] == new_contract)
            ]

            if old_row.empty or new_row.empty:
                continue

            old_price = float(old_row["close"].iloc[0])
            new_price = float(new_row["close"].iloc[0])

            out.loc[idx, "perpetual_price"] = (1 - alpha) * old_price + alpha * new_price
            out.loc[idx, "perpetual_old_weight"] = 1 - alpha
            out.loc[idx, "perpetual_new_weight"] = alpha

    out["perpetual_return"] = out["perpetual_price"].pct_change()

    return out


perpetual_calendar = build_perpetual_weighted_series(
    contracts=contracts,
    raw_series=raw_calendar,
    gap_table=calendar_gap_table,
    roll_window=5
)

perpetual_calendar.head()

## 14. Comparing construction methods

We now compare:

1. raw spliced series;
2. additive back-adjusted series;
3. ratio back-adjusted series;
4. perpetual weighted series.

Each method answers a different question.

| Method | Good for | Main weakness |
|---|---|---|
| Raw splice | Showing actual contract price levels | Artificial jumps at rolls |
| Additive back-adjusted | Trend signals using continuous levels | Can distort/turn negative historical prices |
| Ratio back-adjusted | Percentage-return continuity | Still synthetic, not directly traded |
| Perpetual weighted | Simulating gradual roll exposure | Requires a roll-window trading assumption |

In [ ]:
method_comparison = raw_calendar[["timestamp", "raw_price"]].copy()
method_comparison["back_adjusted_additive"] = additive_calendar["back_adjusted_additive"]
method_comparison["back_adjusted_ratio"] = ratio_calendar["back_adjusted_ratio"]
method_comparison["perpetual_price"] = perpetual_calendar["perpetual_price"]

method_comparison.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(method_comparison["timestamp"], method_comparison["raw_price"], label="Raw splice", alpha=0.75)
plt.plot(method_comparison["timestamp"], method_comparison["back_adjusted_additive"], label="Additive back-adjusted", alpha=0.75)
plt.plot(method_comparison["timestamp"], method_comparison["back_adjusted_ratio"], label="Ratio back-adjusted", alpha=0.75)
plt.plot(method_comparison["timestamp"], method_comparison["perpetual_price"], label="Perpetual weighted", alpha=0.75)
plt.title("Continuous Futures Construction Methods")
plt.xlabel("Date")
plt.ylabel("Synthetic continuous price")
plt.legend()
plt.show()

## 15. Roll-date return diagnostics

If a construction method works as intended, roll-date returns should not contain artificial jumps caused only by switching contract.

We compare the absolute returns on roll dates against non-roll dates.

In [ ]:
diagnostics = pd.DataFrame({
    "timestamp": raw_calendar["timestamp"],
    "is_roll": raw_calendar["is_roll"],
    "raw_return": raw_calendar["raw_return"],
    "additive_return": additive_calendar["additive_return"],
    "ratio_return": ratio_calendar["ratio_return"],
    "perpetual_return": perpetual_calendar["perpetual_return"]
})

diagnostics = diagnostics.dropna().copy()

summary_rows = []

for col in ["raw_return", "additive_return", "ratio_return", "perpetual_return"]:
    roll_abs = diagnostics.loc[diagnostics["is_roll"], col].abs()
    nonroll_abs = diagnostics.loc[~diagnostics["is_roll"], col].abs()

    summary_rows.append({
        "method": col,
        "mean_abs_return_on_roll_dates": roll_abs.mean(),
        "mean_abs_return_non_roll_dates": nonroll_abs.mean(),
        "ratio_roll_to_nonroll": roll_abs.mean() / nonroll_abs.mean()
    })

roll_return_diagnostics = pd.DataFrame(summary_rows)
roll_return_diagnostics

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(
    roll_return_diagnostics["method"],
    roll_return_diagnostics["ratio_roll_to_nonroll"]
)
plt.axhline(1.0, linestyle="--")
plt.title("Roll-Date Absolute Return Inflation by Method")
plt.xlabel("Method")
plt.ylabel("Mean abs roll-date return / mean abs non-roll return")
plt.xticks(rotation=30, ha="right")
plt.show()

## 16. Roll yield diagnostics

The roll gap is not just a data-cleaning nuisance.

It contains information about the futures curve.

For a long futures position, rolling from old to new has approximate roll return:

$$
\frac{F_t^{\text{old}} - F_t^{\text{new}}}{F_t^{\text{old}}}
$$
This is negative in contango and positive in backwardation.

A strategy should distinguish between:

1. **price movement of the held contract**;
2. **return from rolling along the curve**;
3. **synthetic adjustment used for chart continuity**.

In [ ]:
roll_yield_summary = calendar_gap_table.copy()
roll_yield_summary["cumulative_long_roll_return_approx"] = (
    1 + roll_yield_summary["long_roll_return_approx"]
).cumprod() - 1

roll_yield_summary.head()

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(
    roll_yield_summary["timestamp"],
    roll_yield_summary["cumulative_long_roll_return_approx"],
    marker="o"
)
plt.axhline(0, linewidth=1)
plt.title("Cumulative Approximate Roll Return for a Long Position")
plt.xlabel("Roll date")
plt.ylabel("Cumulative approximate roll return")
plt.show()

## 17. Point-in-time safety

A continuous futures series can accidentally introduce look-ahead bias.

Examples:

1. rolling based on future volume;
2. choosing the contract that later became most liquid;
3. using final revised vendor metadata that was unavailable historically;
4. using a roll schedule that depends on later backtest performance;
5. using back-adjusted prices as if they were tradable historical prices.

A safe roll rule should be defined using information available at or before the roll date.

In [ ]:
def validate_continuous_series(
    continuous: pd.DataFrame,
    gap_table: pd.DataFrame,
    price_columns: list[str]
) -> pd.DataFrame:
    """
    Validate basic properties of continuous futures constructions.
    """
    rows = []

    duplicate_dates = int(continuous["timestamp"].duplicated().sum())
    missing_contract_count = int(continuous["selected_contract"].isna().sum())

    for price_col in price_columns:
        missing_prices = int(continuous[price_col].isna().sum())
        non_positive = int((continuous[price_col] <= 0).sum())

        roll_returns = continuous.loc[continuous["is_roll"], price_col].pct_change().dropna()

        rows.append({
            "price_column": price_col,
            "duplicate_timestamps": duplicate_dates,
            "missing_contract_count": missing_contract_count,
            "missing_prices": missing_prices,
            "non_positive_prices": non_positive,
            "num_roll_events": len(gap_table)
        })

    return pd.DataFrame(rows)


validation_table = validate_continuous_series(
    continuous=method_comparison.merge(
        raw_calendar[["timestamp", "selected_contract", "is_roll"]],
        on="timestamp",
        how="left"
    ),
    gap_table=calendar_gap_table,
    price_columns=[
        "raw_price",
        "back_adjusted_additive",
        "back_adjusted_ratio",
        "perpetual_price"
    ]
)

validation_table

## 18. Saving the continuous contract dataset

A useful continuous futures dataset should include metadata about:

- root symbol;
- roll rule;
- selected contract;
- selected expiry;
- raw price;
- adjusted price;
- roll event flag;
- roll gap;
- construction method.

This makes downstream backtests auditable.

In [ ]:
continuous_dataset = raw_calendar[[
    "timestamp",
    "selected_contract",
    "selected_expiry",
    "days_to_expiry",
    "raw_price",
    "raw_return",
    "is_roll",
    "selected_volume",
    "spot_index",
    "carry_rate"
]].copy()

continuous_dataset["additive_back_adjusted_price"] = additive_calendar["back_adjusted_additive"]
continuous_dataset["ratio_back_adjusted_price"] = ratio_calendar["back_adjusted_ratio"]
continuous_dataset["perpetual_price"] = perpetual_calendar["perpetual_price"]

continuous_dataset = continuous_dataset.merge(
    calendar_gap_table[[
        "timestamp",
        "old_contract",
        "new_contract",
        "roll_gap",
        "roll_gap_pct_old",
        "long_roll_return_approx",
        "term_structure_state"
    ]],
    on="timestamp",
    how="left"
)

continuous_dataset["root"] = config.root
continuous_dataset["roll_rule"] = "calendar_5d"
continuous_dataset["schema_version"] = "continuous_futures_v1"
continuous_dataset["generated_at"] = pd.Timestamp.now(tz="UTC")

continuous_dataset.head()

In [ ]:
output_dir = Path("data/processed/continuous_futures_v1")
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "synthetic_continuous_futures_calendar_5d.csv"
continuous_dataset.to_csv(csv_path, index=False)

csv_path

## 19. Limitations

This notebook is deliberately simplified.

### 19.1 Synthetic contracts are cleaner than real futures data

Real futures data includes:

- holidays;
- exchange-specific trading calendars;
- contract listing rules;
- daily price limits;
- delivery notices;
- changing tick sizes;
- changing margin requirements;
- low-liquidity delivery months;
- vendor corrections.

### 19.2 Roll rules are product-specific

A good roll rule for equity index futures may be inappropriate for commodities, rates, FX, or Chinese commodity futures.

Some contracts roll by:

- calendar days before expiry;
- first notice day;
- volume crossover;
- open interest crossover;
- exchange calendar;
- liquidity threshold;
- strategy-specific holding horizon.

### 19.3 Back-adjusted prices are synthetic

Back-adjusted continuous prices are useful for research, but they are not directly tradable historical prices.

This matters when:

- interpreting absolute price levels;
- applying stop losses;
- setting price-based thresholds;
- comparing to spot prices;
- computing carry or roll yield.

### 19.4 Additive adjustment can create negative historical prices

If roll gaps accumulate for long enough, additive back-adjustment can push old prices below zero.

This is especially problematic for commodities with persistent contango.

### 19.5 Ratio adjustment changes the historical scale

Ratio adjustment preserves proportional continuity better, but it still changes historical price levels.

This can affect indicators that depend on absolute prices.

### 19.6 Perpetual series embed a trading assumption

A weighted roll series assumes gradual rolling during the roll window.

That may be realistic for some strategies, but it must be matched with actual execution assumptions and transaction costs.

### 19.7 The correct return series may not be the same as the correct chart series

A visually smooth price chart is not always the correct object for return attribution.

In futures, strategy PnL should be based on the actual contracts held, roll execution, contract multipliers, margin, and transaction costs.

## 20. Research frontier and current directions

Continuous futures construction is still an active practical problem because the method affects backtests, risk estimates, and signal design.

### 20.1 Point-in-time continuous futures datasets

Modern research pipelines increasingly separate:

- raw contract data;
- roll schedules;
- adjusted research series;
- tradable return series.

This separation helps avoid treating adjusted chart history as if it were directly executable.

A future notebook could store roll schedules as first-class datasets and replay the exact contracts held by a strategy.

### 20.2 Curve-aware modelling instead of single continuous prices

A single continuous contract compresses the futures curve into one time series.

For commodities and rates, important information lives in the curve shape:

- front/back spread;
- calendar spreads;
- slope;
- curvature;
- carry;
- seasonality;
- inventory pressure.

A future notebook could model the full term structure using PCA or dynamic factor models instead of relying only on the front continuous contract.

### 20.3 Roll yield decomposition

Trend-following futures strategies often combine spot-price movement and roll/carry effects.

A more advanced pipeline should decompose returns into:

1. spot or nearby-price movement;
2. roll yield;
3. collateral return;
4. transaction costs;
5. slippage.

A future notebook could build an excess-return futures index and compare it with a price-only continuous contract.

### 20.4 Liquidity-aware and execution-aware rolling

Real funds care about volume, open interest, bid-ask spread, market impact, and roll crowding.

A roll rule that is good for charts may be bad for execution.

A future notebook could simulate roll execution costs and compare calendar rolling with volume-crossover rolling.

### 20.5 Futures curve no-arbitrage and storage models

Commodity futures curves are linked to storage costs, convenience yield, financing rates, seasonality, and inventory constraints.

A future notebook could model curve dynamics under a cost-of-carry framework and test whether synthetic curves violate basic no-arbitrage relationships.

### 20.6 Exchange-specific microstructure

Chinese commodity futures introduce additional practical details:

- night sessions;
- exchange-specific contract codes;
- daily limit-up and limit-down rules;
- margin schedule changes;
- delivery-month restrictions;
- retail and institutional participation differences.

A future notebook could extend the schema to handle DCE, CZCE, SHFE, and INE contract metadata.

## 21. Suggested follow-up notebooks

This notebook naturally leads to:

1. **01_04_return_sanitization_and_bias_control**  
   Checking whether continuous-contract returns contain artificial jumps, leakage, or adjustment artefacts.

2. **01_07_futures_calendar_and_roll_yield_decomposition**  
   Turning roll gaps into carry and roll-yield features.

3. **03_10_statistical_arbitrage_pairs**  
   Using futures calendar spreads and cross-contract relationships.

4. **04_08_futures_minimum_variance_hedge_ratio**  
   Hedging spot or equity exposure using futures contracts.

5. **05_01_vectorized_backtest_engine**  
   Backtesting strategies on actual rolled contract holdings rather than only adjusted charts.

6. **06_07_limit_board_margin_and_deadband_rebalancing**  
   Adding exchange-specific frictions and margin rules for Chinese futures.

7. **07_01_multi_asset_cta_research_pipeline**  
   Integrating futures rolling into an end-to-end CTA research workflow.

## 22. Summary

This notebook constructed continuous futures series from individual expiring contracts.

The main steps were:

1. simulate individual futures contracts;
2. define calendar and volume roll schedules;
3. build a raw spliced continuous series;
4. measure roll gaps;
5. apply additive backward adjustment;
6. apply ratio backward adjustment;
7. build a perpetual weighted roll series;
8. diagnose roll-date return inflation;
9. estimate approximate roll yield;
10. save an auditable continuous futures dataset.

The key computational takeaway is:

> A continuous futures contract is not raw data. It is a modelling choice created by a roll rule and an adjustment method.

The key financial takeaway is:

> The correct construction depends on the question. A smooth chart, a tradable return series, and a roll-yield decomposition are related but not identical objects.

## 23. Further reading

### Continuous futures construction

- CME Group continuous price series methodology.
- CME Group education material on futures expiration and contract rolls.
- Research papers and practitioner notes on continuous futures contracts for backtesting.
- QuantStart and Quantpedia articles on Panama, proportional, and perpetual adjustment methods.
- Vendor documentation for back-adjusted futures charts.

### Futures returns and roll yield

- CME Group material on deconstructing futures returns and roll yield.
- Commodity futures literature on contango, backwardation, storage, and convenience yield.
- Managed futures and trend-following literature on excess returns and collateral returns.

### Future extensions

- Point-in-time futures metadata.
- Calendar-spread and curve factor models.
- Liquidity-aware roll scheduling.
- Execution-aware roll simulation.
- Contract-specific backtesting and PnL attribution.